

O dicionário original do Kaggle (`Data_Dictionary.xls`) e os schemas que fomos ajustando no Silver e no Gold. Consolidei abaixo a versão mais recente, que é de 11/09/2026 (conversas "Adequação do projeto risco_credito_databricks às melhores práticas" e "Levantamento de modelo para projeto").

## Colunas de negócio (Kaggle → Silver → Gold)

| Kaggle (Bronze) | Silver | Gold | Descrição |
|---|---|---|---|
| (índice) | `customer_id` | `customer_id` (dim e fct) | Identificador do cliente, chave de negócio |
| `SeriousDlqin2yrs` | `target_default_2yrs` | `target_dlq_2yrs` | Alvo: 1 = atraso de 90+ dias em 2 anos, 0 = não. **Nulo nas linhas de scoring** |
| `RevolvingUtilizationOfUnsecuredLines` | `revolving_utilization_unsecured` | `revolving_utilization` | Saldo em cartões e linhas pessoais dividido pelos limites |
| `age` | `age` | `age` (dim) | Idade do cliente |
| `NumberOfTime30-59DaysPastDueNotWorse` | `num_times_30_59_days_late` | `num_times_30_59_days_late` | Vezes com atraso de 30 a 59 dias |
| `DebtRatio` | `debt_ratio` | `debt_ratio` | Pagamentos mensais de dívidas e custos divididos pela renda bruta mensal |
| `MonthlyIncome` | `monthly_income` | `monthly_income` | Renda mensal |
| `NumberOfOpenCreditLinesAndLoans` | `num_open_credit_lines_and_loans` | `num_open_credit_lines` | Nº de empréstimos e linhas de crédito abertos |
| `NumberOfTimes90DaysLate` | `num_times_90_days_late` | `num_times_90_days_late` | Vezes com atraso de 90+ dias |
| `NumberRealEstateLoansOrLines` | `num_real_estate_loans_or_lines` | `num_real_estate_loans` | Nº de financiamentos e linhas imobiliárias |
| `NumberOfTime60-89DaysPastDueNotWorse` | `num_times_60_89_days_late` | `num_times_60_89_days_late` | Vezes com atraso de 60 a 89 dias |
| `NumberOfDependents` | `num_dependents` | `num_dependents` (dim) | Nº de dependentes; nulo vira 0 no Silver |

## Colunas derivadas e de controle

| Coluna | Camada | Descrição |
|---|---|---|
| `is_monthly_income_null` | Silver | Flag 1/0 para renda nula. Não vai para o Gold |
| `has_delinquency_outlier` | Silver | Flag 1/0 quando qualquer coluna de atraso é ≥ 96. Não vai para o Gold |
| `total_delinquency_events` | Gold (`fct_credit_profile`) | Soma das três colunas de atraso (feature derivada) |
| `_ingestion_timestamp`, `_source_file` | Bronze e Silver | Linhagem da ingestão |
| `_silver_processed_at` | Silver | Timestamp de processamento do Silver |
| `_gold_processed_at` | Gold (`dim_customer`) | Timestamp de processamento do Gold |

## Estrutura Gold

- **`dim_customer`**: `customer_id`, `age`, `num_dependents`, `_gold_processed_at`
- **`fct_credit_profile`**: `customer_id`, as 6 variáveis de crédito e atraso, `total_delinquency_events` e `target_dlq_2yrs`

## Regras de limpeza no Silver

- `age` fora de 18–115 vira nulo.
- `num_dependents` nulo vira 0.
- O Silver e o Gold deduplicam por `customer_id`.

## Pontos de atenção

- **Descrições:** vêm do dicionário padrão do Kaggle, não do seu `.xls`, que não consigo reabrir daqui.
- **Regras de limpeza:** vêm da versão de 09/09 do `silver.py`. Na versão consolidada que você subiu em 11/09 eu não vi o código, só a menção às mesmas duas flags.
- **Catálogo:** hoje o pipeline roda em `credito_dev`; o `credito_prd` aparece nas conversas anteriores.

Se quiser, transformo isso em `docs/dicionario_de_dados.md` ou em `COMMENT` por coluna para o Unity Catalog.